# Construcción del modelo 1D-CNN

En este cuaderno se construye y se verifica la arquitectura de red neuronal convolucional unidimensional (1D-CNN) que se usará para clasificar presencia o ausencia de enfermedad cardíaca, tomando como fuente el artículo semilla (Honi y Szathmary, 2024). Solo se construye el modelo: no se entrena, no se evalúa y no se usa K-Fold.

## 1. Importación de TensorFlow y Keras

Se importa TensorFlow y se comprueba la versión instalada, que debe ser la misma del entorno del proyecto.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # reduce mensajes informativos de TensorFlow

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)

TensorFlow: 2.21.0
Keras: 3.15.1


**Interpretación del resultado**

TensorFlow 2.21.0 se importó sin errores, la versión con la que se configuró el entorno. Keras viene incluido y es la interfaz con la que se definirán las capas. Se importan también numpy y pandas, solo para leer y dar forma a los datos.

## 2. Preparación de los datos de entrada

Se leen los conjuntos escalados guardados en el Notebook 02 (1025 registros) y se les agrega una dimensión final de tamaño 1, que es el formato que pide una capa Conv1D.

In [2]:
carpeta = "../resultados/csv/"

X_train = pd.read_csv(carpeta + "X_train.csv").values.astype("float32")
X_test = pd.read_csv(carpeta + "X_test.csv").values.astype("float32")
y_train = pd.read_csv(carpeta + "y_train.csv").values.ravel()
y_test = pd.read_csv(carpeta + "y_test.csv").values.ravel()

print("Antes (registros, variables):", X_train.shape, X_test.shape)

# Una CNN 1D espera (registros, posiciones, canales): cada variable es una posición y hay un solo canal
X_train = X_train.reshape(-1, 13, 1)
X_test = X_test.reshape(-1, 13, 1)

print("Después (registros, variables, canal):", X_train.shape, X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Rango de X_train: de", X_train.min(), "a", X_train.max())
print("Rango de X_test:  de", X_test.min(), "a", X_test.max())
print("Total de registros:", len(X_train) + len(X_test))

Antes (registros, variables): (717, 13) (308, 13)
Después (registros, variables, canal): (717, 13, 1) (308, 13, 1)
y_train: (717,) | y_test: (308,)
Rango de X_train: de 0.0 a 1.0
Rango de X_test:  de 0.0 a 1.0
Total de registros: 1025


**Interpretación del resultado**

Los datos pasaron de (717, 13) y (308, 13) a (717, 13, 1) y (308, 13, 1), lo que suma los 1025 registros del flujo principal. El número de variables no cambió: la dimensión final `1` es el canal, como una imagen en escala de grises tiene un solo canal. Con esta forma la CNN recorre las 13 variables como si fueran 13 posiciones consecutivas de una secuencia. Los valores siguen en el rango [0, 1] del escalado.

## 3. Arquitectura descrita en el artículo

Antes de construir el modelo se resume lo que el artículo indica (Figura 7, Tabla 2 y texto de la sección 3.5). Solo se incluyen datos que aparecen en el artículo.

| Orden | Capa | Parámetros indicados en el artículo |
|---|---|---|
| 1 | Entrada | Forma (13, 1) |
| 2 | Conv1D, bloque 1 | 32 filtros, kernel 2, activación sigmoid |
| 3 | BatchNormalization | Sin parámetros indicados |
| 4 | Dropout | 0.2 |
| 5 | Conv1D, bloque 2 | 64 filtros, kernel 2, activación sigmoid |
| 6 | BatchNormalization | Sin parámetros indicados |
| 7 | Dropout | 0.5 |
| 8 | Conv1D, bloque 3 | 128 filtros, kernel 2, activación sigmoid |
| 9 | BatchNormalization | Sin parámetros indicados |
| 10 | Conv1D, bloque 4 | 128 filtros, kernel 2, activación sigmoid |
| 11 | BatchNormalization | Sin parámetros indicados |
| 12 | Flatten | Figura 7: 1152 valores. Tabla 2: 1280 valores |
| 13 | Dense | 200 neuronas, activación ReLU |
| 14 | Dropout | 0.1 |
| 15 | Dense de salida | 1 neurona, activación sigmoid |

El artículo **no describe ninguna capa de pooling**. El orden dentro de cada bloque (Conv1D, BatchNormalization, Dropout) se toma de la Tabla 2. La Figura 7 y la Tabla 2 no coinciden del todo: la Figura 7 da 32, 64, 128 y 128 filtros con kernel 2, mientras que los parámetros de la Tabla 2 en las capas convolucionales segunda y tercera no corresponden a esos filtros, y el Flatten es 1280 en la tabla y 1152 en la figura. Se usa la Figura 7 como fuente principal y la diferencia se mide más adelante.

## 4. Creación del modelo

Se construye la arquitectura con `Sequential`, capa por capa, siguiendo la tabla anterior.

In [3]:
tf.keras.utils.set_random_seed(42)   # semilla fijada por el proyecto, no por el artículo

model = tf.keras.Sequential([
    layers.Input(shape=(13, 1)),

    # Bloque 1
    layers.Conv1D(filters=32, kernel_size=2, activation="sigmoid"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    # Bloque 2
    layers.Conv1D(filters=64, kernel_size=2, activation="sigmoid"),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    # Bloque 3
    layers.Conv1D(filters=128, kernel_size=2, activation="sigmoid"),
    layers.BatchNormalization(),

    # Bloque 4
    layers.Conv1D(filters=128, kernel_size=2, activation="sigmoid"),
    layers.BatchNormalization(),

    # Parte final: clasificación
    layers.Flatten(),
    layers.Dense(200, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(1, activation="sigmoid"),
])

print("Modelo creado")
print("Número de capas (sin contar la entrada):", len(model.layers))

Modelo creado
Número de capas (sin contar la entrada): 14


**Interpretación del resultado**

El modelo se creó sin errores y tiene 14 capas sin contar la entrada: 4 convolucionales, 4 de normalización por lotes, 3 de dropout, 1 Flatten y 2 densas. Se usó `Sequential` porque la información pasa de una capa a la siguiente en línea recta, sin ramas. Cada argumento corresponde a un dato de la tabla anterior; el relleno (`padding`) quedó con el valor por defecto de Keras porque el artículo no lo indica.

## 5. Resumen del modelo

Con `model.summary()` se muestra cada capa, su forma de salida y su cantidad de parámetros.

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 12, 32)         │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 12, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 11, 64)         │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 11, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 11, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 10, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 9, 128)         │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 9, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 200)            │       230,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 285,873 (1.09 MB)

 Trainable params: 285,169 (1.09 MB)

 Non-trainable params: 704 (2.75 KB)

**Interpretación del resultado**

La entrada (13, 1) se convierte en (12, 32), (11, 64), (10, 128) y (9, 128) al pasar por las cuatro convoluciones: cada una acorta la secuencia en 1 porque el kernel mide 2, y aumenta el número de filtros. El Flatten convierte (9, 128) en un vector de 1152 valores, la capa densa lo reduce a 200 y la salida final es 1 valor. El modelo tiene 285,873 parámetros, de los cuales 285,169 son entrenables y 704 no lo son (son estadísticas internas de la normalización por lotes). La capa densa de 200 neuronas concentra la mayor parte: 230,600.

## Comparación con la Tabla 2 del artículo

Se comparan los parámetros por capa del modelo construido con los de la Tabla 2, para ver exactamente dónde hay diferencias.

In [5]:
# Valores de la Tabla 2 del artículo (página 8)
tabla2 = {
    "Conv1D 1": 96, "BatchNorm 1": 128,
    "Conv1D 2": 2080, "BatchNorm 2": 256,
    "Conv1D 3": 8256, "BatchNorm 3": 512,
    "Conv1D 4": 32896, "BatchNorm 4": 512,
    "Dense 200": 256200, "Dense salida": 201,
}

capas_con_pesos = [c for c in model.layers if c.count_params() > 0]
comparacion = pd.DataFrame({
    "Capa": list(tabla2.keys()),
    "Tabla 2 del artículo": list(tabla2.values()),
    "Modelo construido": [c.count_params() for c in capas_con_pesos],
})
comparacion["Coincide"] = np.where(
    comparacion["Tabla 2 del artículo"] == comparacion["Modelo construido"], "Sí", "No"
)

print(comparacion.to_string(index=False))
print()
print("Total según la suma de la Tabla 2:", sum(tabla2.values()))
print("Total del modelo construido:      ", model.count_params())
print("Flatten: Tabla 2 = 1280, Figura 7 = 1152, modelo =", model.layers[-4].output.shape[-1])

        Capa  Tabla 2 del artículo  Modelo construido Coincide
    Conv1D 1                    96                 96       Sí
 BatchNorm 1                   128                128       Sí
    Conv1D 2                  2080               4160       No
 BatchNorm 2                   256                256       Sí
    Conv1D 3                  8256              16512       No
 BatchNorm 3                   512                512       Sí
    Conv1D 4                 32896              32896       Sí
 BatchNorm 4                   512                512       Sí
   Dense 200                256200             230600       No
Dense salida                   201                201       Sí

Total según la suma de la Tabla 2: 301137
Total del modelo construido:       285873
Flatten: Tabla 2 = 1280, Figura 7 = 1152, modelo = 1152


**Interpretación del resultado**

Coinciden 7 de las 10 capas con parámetros: las normalizaciones por lotes, la primera y la cuarta convolución y la capa de salida. Difieren las convoluciones 2 y 3 y la capa densa de 200 neuronas. El Flatten del modelo es 1152, igual que la Figura 7 y distinto de los 1280 de la Tabla 2, y de ahí viene la diferencia en la capa densa. Los totales también difieren (301,137 en la suma de la Tabla 2 y 285,873 en el modelo). Como la Tabla 2 no es coherente con la Figura 7, la reproducción exacta de esos números no es posible y debe declararse como una limitación.

## 6. Compilación

Se definen el optimizador, la función de pérdida y la métrica. Compilar prepara el modelo para entrenar, pero no entrena nada.

In [6]:
# El artículo no indica optimizador, tasa de aprendizaje, pérdida ni métricas de compilación.
# Los siguientes valores son decisiones del proyecto para la reproducción.
metricas = ["accuracy"]
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=metricas,
)

print("Optimizador:", type(model.optimizer).__name__)
print("Learning rate:", float(model.optimizer.learning_rate.numpy()))
print("Función de pérdida:", model.loss)
print("Métricas:", metricas)

Optimizador: Adam
Learning rate: 0.0010000000474974513
Función de pérdida: binary_crossentropy
Métricas: ['accuracy']


**Interpretación del resultado**

El modelo quedó compilado con Adam, tasa de aprendizaje 0.001 y pérdida `binary_crossentropy`, adecuada para una salida binaria con sigmoid. **Ninguno de estos valores aparece en el artículo**: son decisiones del proyecto y así deben presentarse. La métrica `accuracy` se eligió porque el artículo reporta exactitud en sus resultados. Los pesos todavía no se han modificado.

## 7. Verificación de la arquitectura

Se comprueba que el modelo acepte un lote pequeño de datos y devuelva una salida con la forma correcta, sin entrenar ni cambiar los pesos.

In [7]:
pesos_antes = [w.copy() for w in model.get_weights()]

lote = X_train[:8]
predicciones = model.predict(lote, verbose=0)

pesos_despues = model.get_weights()
sin_cambios = all(np.array_equal(a, b) for a, b in zip(pesos_antes, pesos_despues))

print("Forma de entrada del modelo:", model.input_shape)
print("Forma de salida del modelo: ", model.output_shape)
print("Forma del lote de prueba:   ", lote.shape)
print("Forma de las predicciones:  ", predicciones.shape)
print("Valor mínimo y máximo de la salida: {:.4f} y {:.4f}".format(predicciones.min(), predicciones.max()))
print("Salidas dentro de (0, 1):", bool(((predicciones > 0) & (predicciones < 1)).all()))
print("Pesos iguales antes y después de predecir:", sin_cambios)

prueba = model.predict(X_test[:8], verbose=0)
print("Predicción sobre un lote de X_test:", prueba.shape)

Forma de entrada del modelo: (None, 13, 1)
Forma de salida del modelo:  (None, 1)
Forma del lote de prueba:    (8, 13, 1)
Forma de las predicciones:   (8, 1)
Valor mínimo y máximo de la salida: 0.8128 y 0.8129
Salidas dentro de (0, 1): True
Pesos iguales antes y después de predecir: True
Predicción sobre un lote de X_test: (8, 1)


**Interpretación del resultado**

El modelo recibe entradas (None, 13, 1) y devuelve salidas (None, 1): para 8 registros entrega 8 valores, cada uno entre 0 y 1. Los dos lotes, uno de entrenamiento y otro de prueba, se procesaron sin errores y los pesos no cambiaron, lo que confirma que solo se hizo una predicción y no hubo entrenamiento. Los valores de salida son casi iguales entre sí porque la red aún no aprendió nada, así que **no significan nada clínico**.

## 8. Relación entre la arquitectura y el problema

Se sigue la forma de los datos capa por capa para ver qué hace cada parte de la red con las 13 variables de un paciente.

In [8]:
print("Entrada:", model.input_shape[1:])
for capa in model.layers:
    if "conv1d" in capa.name or "flatten" in capa.name or "dense" in capa.name:
        print(f"{capa.name:10s} -> {capa.output.shape[1:]}")

Entrada: (13, 1)
conv1d     -> (12, 32)
conv1d_1   -> (11, 64)
conv1d_2   -> (10, 128)
conv1d_3   -> (9, 128)
flatten    -> (1152,)
dense      -> (200,)
dense_1    -> (1,)


**Interpretación del resultado**

Cada paciente entra como 13 valores con un canal. **Conv1D** desliza un filtro de tamaño 2 sobre pares de variables vecinas y genera varios mapas de características, por eso la longitud baja de 13 a 9 y los canales suben a 128. **Flatten** convierte esos mapas en un vector de 1152 valores, y las **capas densas** combinan ese vector: la primera con 200 neuronas y la última con una sola neurona sigmoid cuya salida es una probabilidad entre 0 y 1 de presencia de enfermedad cardíaca (clasificación binaria). El artículo no incluye capas de pooling, por lo que aquí tampoco hay. Como Conv1D mira variables contiguas, el orden de las columnas influye en qué pares se combinan.

## 9. Resumen del modelo

Se construye una tabla final con la configuración real del modelo, leída directamente de las capas.

In [9]:
convoluciones = [c for c in model.layers if isinstance(c, layers.Conv1D)]
dropouts = [c.rate for c in model.layers if isinstance(c, layers.Dropout)]
densas = [c for c in model.layers if isinstance(c, layers.Dense)]

resumen = pd.DataFrame([
    ("Forma de entrada", str(model.input_shape[1:])),
    ("Forma de salida", str(model.output_shape[1:])),
    ("Capas (sin la entrada)", len(model.layers)),
    ("Capas Conv1D", len(convoluciones)),
    ("Filtros por Conv1D", [c.filters for c in convoluciones]),
    ("Tamaño del kernel", [c.kernel_size[0] for c in convoluciones]),
    ("Activación Conv1D", sorted({c.activation.__name__ for c in convoluciones})),
    ("Padding Conv1D", sorted({c.padding for c in convoluciones})),
    ("BatchNormalization", sum(isinstance(c, layers.BatchNormalization) for c in model.layers)),
    ("Tasas de Dropout", dropouts),
    ("Capas de pooling", sum("pooling" in type(c).__name__.lower() for c in model.layers)),
    ("Neuronas Dense (oculta)", densas[0].units),
    ("Activación Dense (oculta)", densas[0].activation.__name__),
    ("Neuronas de salida", densas[1].units),
    ("Activación de salida", densas[1].activation.__name__),
    ("Parámetros totales", f"{model.count_params():,}"),
    ("Parámetros entrenables", f"{sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}"),
    ("Parámetros no entrenables", f"{sum(int(np.prod(w.shape)) for w in model.non_trainable_weights):,}"),
    ("Optimizador / learning rate", "Adam / 0.001 (decisión del proyecto)"),
    ("Función de pérdida", "binary_crossentropy (decisión del proyecto)"),
], columns=["Elemento", "Configuración"])

resumen

,Elemento,Configuración
0,Forma de entrada,"(13, 1)"
1,Forma de salida,"(1,)"
2,Capas (sin la entrada),14
3,Capas Conv1D,4
4,Filtros por Conv1D,"[32, 64, 128, 128]"
5,Tamaño del kernel,"[2, 2, 2, 2]"
6,Activación Conv1D,[sigmoid]
7,Padding Conv1D,[valid]
8,BatchNormalization,4
9,Tasas de Dropout,"[0.2, 0.5, 0.1]"


**Interpretación del resultado**

La tabla confirma que el modelo tiene entrada (13, 1), cuatro Conv1D con 32, 64, 128 y 128 filtros de kernel 2 y activación sigmoid, tres capas de dropout con tasas 0.2, 0.5 y 0.1, una densa de 200 neuronas con ReLU y una salida sigmoid de una neurona. No hay capas de pooling. El `padding` quedó en `valid`, valor por defecto que no proviene del artículo. Estos datos salen del propio objeto `model`, no fueron escritos a mano.

## Parámetros identificados en el artículo

Se separan en tres grupos: lo que el artículo indica, lo que decidió el proyecto y lo que no se especifica.

**1. Parámetros que aparecen explícitamente en el artículo**

- Entrada de la red con forma (13, 1).
- Cuatro bloques convolucionales con 32, 64, 128 y 128 filtros y kernel de tamaño 2 (Figura 7).
- Activación sigmoid en las capas convolucionales (Figura 7 y Tabla 2).
- Una capa BatchNormalization después de cada convolución (Figura 7 y Tabla 2).
- Dropout de 0.2 tras el bloque 1 y de 0.5 tras el bloque 2 (Figura 7 y texto).
- Dropout de 0.1, ubicado después de la capa densa en la Figura 7 y la Tabla 2. El texto dice que los tres dropout van "después de cada capa convolucional", lo cual contradice esa ubicación.
- Orden de capas dentro de cada bloque: convolución, normalización, dropout (Tabla 2).
- Capa Flatten, capa densa de 200 neuronas con ReLU y capa de salida de 1 neurona con sigmoid.
- Entrenamiento durante 150 épocas y división 70/30 (todavía no aplicados en este notebook).

**2. Parámetros establecidos por el proyecto para la reproducibilidad**

- Optimizador Adam con tasa de aprendizaje 0.001.
- Función de pérdida `binary_crossentropy` y métrica `accuracy`.
- Semilla aleatoria 42.
- `padding="valid"` (valor por defecto de Keras), elegido porque produce un Flatten de 1152 valores, igual al de la Figura 7.
- Uso de 13 variables sin One-Hot y de los 1025 registros con duplicados (decisiones del Notebook 02).
- Los datos de tipo `float32` y el uso de `Sequential`.

**3. Parámetros que el artículo no especifica**

- Optimizador, tasa de aprendizaje, función de pérdida y métricas de compilación.
- Tamaño del lote (batch size).
- Tipo de relleno (padding) y paso (stride) de las convoluciones.
- Parámetros internos de BatchNormalization (momentum y epsilon).
- Inicialización de pesos y semilla aleatoria.
- Uso de pooling: no se describe ninguna capa de pooling.
- Si se usó un conjunto de validación dentro del entrenamiento, early stopping o regularización.
- Cuál de los dos tamaños de Flatten (1152 o 1280) es el correcto y qué configuración produce los parámetros de las convoluciones 2 y 3 de la Tabla 2.

## Conclusión de esta fase

La arquitectura 1D-CNN fue construida con Keras siguiendo la Figura 7 del artículo, con entrada (13, 1) para las 13 variables clínicas, y quedó compilada y verificada con lotes de prueba. Tiene 285,873 parámetros y la Tabla 2 del artículo no coincide completamente con ella (capas convolucionales 2 y 3 y capa densa). El optimizador, la tasa de aprendizaje y la función de pérdida son decisiones del proyecto, porque el artículo no las indica. El modelo está preparado para entrenarse, pero todavía no se ha entrenado ni evaluado. El siguiente paso es el Notebook 04, con el entrenamiento y la evaluación.